# 🧠 Colab 1: Data Preparation & Classical Baseline

This notebook covers **Phase 2** of the project: setting up the environment, loading the dataset, and training the classical EfficientNet-B0 baseline.

## 1. Environment Setup

In [ ]:
!pip install pennylane --quiet
!pip install torch torchvision torchaudio --quiet
!pip install matplotlib seaborn scikit-learn pandas numpy --quiet

## 2. Dataset Loading

This section handles loading the dataset from **Google Drive** (as a zip file) or from a local directory.

**Ideally:** Upload `dataset.zip` to your Google Drive root or a specific folder.

In [ ]:
import os
import shutil

# --- Configuration ---
# Path to the zip file in Google Drive.
# User should change this to match their Drive location.
DRIVE_ZIP_PATH = '/content/drive/MyDrive/dataset.zip' 
DATASET_DIR = '/content/dataset' # Local Colab path to unzip to

# --- 1. Mount Google Drive ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted.")
except ImportError:
    print("⚠️ Not running in Google Colab or Drive not mounted.")
except Exception as e:
    print(f"⚠️ Drive mount skipped or failed: {e}")

# --- 2. Copy & Unzip Dataset ---
if os.path.exists(DRIVE_ZIP_PATH):
    print(f"📦 Found zip at {DRIVE_ZIP_PATH}. Copying to local runtime...")
    # Copy to local runtime for speed
    shutil.copy(DRIVE_ZIP_PATH, '/content/dataset.zip')
    
    print("📂 Unzipping dataset...")
    # Unzip
    shutil.unpack_archive('/content/dataset.zip', DATASET_DIR)
    print(f"✅ Dataset ready at {DATASET_DIR}")
    
    # Update data_dir to point to the unzipped location
    # structure is usually dataset/Training
    data_dir = os.path.join(DATASET_DIR, 'Training') 

elif os.path.exists('../data/Training'):
    # Fallback to local data (for local development)
    print("✅ Found local data in ../data/Training")
    data_dir = '../data/Training'

else:
    print(f"❌ Dataset not found at {DRIVE_ZIP_PATH} or ../data/Training")
    print("Please upload 'dataset.zip' to your Drive or place data in the local folder.")
    # Set a default just to avoid immediate crash in next cell if user fixes it manually
    data_dir = './dataset/Training'

## 3. Data Preprocessing & Loading

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print(f"📂 Loading data from: {data_dir}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

try:
    dataset = datasets.ImageFolder(data_dir, transform=transform)
    print(f"✅ Classes detected: {dataset.classes}")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")

## 4. Classical Baseline (EfficientNet-B0)

In [ ]:
import torchvision.models as models
import torch.nn as nn

def get_classical_model(num_classes=4):
    model = models.efficientnet_b0(pretrained=True)
    
    # Freeze feature extractor
    for param in model.features.parameters():
        param.requires_grad = False
        
    # Replace classifier head
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

model = get_classical_model()
print(model)